In [134]:
import pickle as pkl
import numpy as np
from pathlib import Path
import tarfile
import glob
import re
from collections import defaultdict, Counter
import itertools
import matplotlib.pyplot as plt
import pandas as pd
# import dionysus

# from helper_functions import *
# from KK_zz_apex_LS import *

from hierarchicalsoftmax import SoftmaxNode, HierarchicalSoftmaxLoss, HierarchicalSoftmaxLinear
from hierarchicalsoftmax.inference import (
    greedy_predictions,
    node_probabilities,
)

import numpy as np
from collections import defaultdict, Counter
import itertools
import pandas as pd
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim


# Load the pickle files from Kritika and save for zigzag PH

{sequenc: {'sequence_id':id, 'mapped_label':label}}

Stratified 5-fold split based on superfamily labels
* if only order provided, then superfamily==order for sake of split

<80bp length removed

'>5% non atgc removed

**Should also remove if n superfamily < 5**

In [151]:
def process_pkl(mapped_data):
    # Save sequence and label
    sequences = []
    labels = []
    for k, value in mapped_data.items():
        sequences.append(k.lower())
        labels.append(value['mapped_label'])

    seq_x = [re.sub(r'[^atcg]', 'x', seq) for seq in sequences]

    # Create df, split labels into order and superfamily
    seq_df = pd.DataFrame({'sequence':sequences, 'seq_x':seq_x, 'labels':labels})
    seq_df[['order', 'superfamily']] = seq_df['labels'].str.split('/',expand=True) #split
    seq_df['superfamily'] = seq_df['superfamily'].fillna(seq_df['order']) # if only order, superfamily==order
    return seq_df, sequences, labels

def kfold_strat_split(seq_df, k=5, seed=7):
    # Split 
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    X = np.array(list(seq_df['seq_x']))
    # y = np.array(labels)
    y = np.array(list(seq_df['superfamily']))

    for fold_index, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        col_name = f'fold_{fold_index}'
        seq_df[col_name] = 'train' # Default everyone to train
        seq_df.loc[val_idx, col_name] = 'test'

    return seq_df

def get_seq_stats(seq_df):
    seq_df['seq_length'] = seq_df['seq_x'].str.len() # seq length
    seq_df['n_non_actg'] = seq_df['seq_x'].str.count('[^actg]') # number non canonical
    seq_df['frac_non_actg'] = seq_df['n_non_actg'] / seq_df['seq_length'] # fraction non canonical
    return seq_df

def filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2):
    seq_df_filt = seq_df[(seq_df['seq_length'] >= len_thresh) & (seq_df['frac_non_actg'] <= frac_thresh)]
    # Remove any labels with <5 sequences
    label_counts = Counter(seq_df_filt['labels'])
    to_remove = {label for label, count in label_counts.items() if count < 5}
    seq_df_filt = seq_df_filt[~seq_df_filt['labels'].isin(to_remove)]
    print(f"Removing these labels: {to_remove}")
    return seq_df_filt.reset_index()

def save_fasta(filename, seq_df):
    sequences = list(seq_df['seq_x'])
    labels = list(seq_df['labels'])
    with open(filename, "w") as f:
        for i, (label, seq) in enumerate(zip(labels, sequences)):
            f.write(f">{i}_{label}\n")
            f.write(f"{seq.lower()}\n")
    print(f"Saved here: {filename}")

def split_fasta(input_file, path, chunk_size=1000):
    with open(input_file, 'r') as f:
        sequences = []
        current_seq = []
        file_count = 1
        
        for line in f:
            line = line.strip()
            if not line: continue
            
            if line.startswith('>'):
                # If we have a sequence buffered, join it and store it
                if current_seq:
                    sequences.append("".join(current_seq))
                    current_seq = []
                
                # Check if we need to save the current chunk
                if len(sequences) >= chunk_size:
                    save_chunk(sequences, path, file_count)
                    sequences = []
                    file_count += 1
            else:
                current_seq.append(line)

        # Handle the very last sequence in the file
        if current_seq:
            sequences.append("".join(current_seq))
        
        # Save the final remaining chunk
        if sequences:
            save_chunk(sequences, path, file_count)

def save_chunk(data, path, count):
    output_name = f"{path}/chunk_{count}.txt"
    with open(output_name, 'w') as out:
        out.write("\n".join(data) + "\n")
    # print(f"Saved {output_name}")

### Repbase

In [153]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "RepBase31pt04_converted2_terrierlabelsMay13.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../repbase_QC_051326.fasta'
save_path = '../Repbase/'

csv_save = '../repbase_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)

# Save df 
# seq_df.to_csv(csv_save, index=False)

# Save big fasta
save_fasta(filename, seq_df)

# Save chunks
output_dir = Path(save_path)
output_dir.mkdir(parents=True, exist_ok=True)
split_fasta(filename, save_path, 1000)


Removing these labels: set()
Saved here: ../repbase_QC_051326.fasta


### RepetDB

In [152]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "Repetdb_allfiltered_terrier_May13.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../repetdb_QC_051326.fasta'
save_path = '../RepetDB/'

csv_save = '../repetdb_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)

# Save df 
# seq_df.to_csv(csv_save, index=False)

# Save big fasta
save_fasta(filename, seq_df)

# Save chunks
output_dir = Path(save_path)
output_dir.mkdir(parents=True, exist_ok=True)
split_fasta(filename, save_path, 1000)

Removing these labels: {'RC/Helitron'}
Saved here: ../repetdb_QC_051326.fasta


### MnTEdb

In [154]:
# Load data
pkl_path = "./../TERRIER_labellingsystems_alldatasets"
dataset = "MnTEdb_terrier.pkl"
db_path = f"{pkl_path}/{dataset}"

filename = '../mntedb_QC_051326.fasta'
save_path = '../MnTEdb/'

csv_save = '../mntedb_QC_051326.csv'

with open(db_path, "rb") as f:
    mapped_data = pkl.load(f)

seq_df, sequences, labels = process_pkl(mapped_data)

# Filter
seq_df = get_seq_stats(seq_df)
seq_df = filter_seqs(seq_df, len_thresh=80, frac_thresh=0.2)

# Stratified split
seq_df = kfold_strat_split(seq_df, k=5, seed=7)

# Save df 
# seq_df.to_csv(csv_save, index=False)

# Save big fasta
save_fasta(filename, seq_df)

# Save chunks
output_dir = Path(save_path)
output_dir.mkdir(parents=True, exist_ok=True)
split_fasta(filename, save_path, 1000)

Removing these labels: set()
Saved here: ../mntedb_QC_051326.fasta
